# OpenTelemetry Traces: Seeing the Full Agent → Cycle → LLM → Tool Tree

Demo 01 showed `result.metrics.get_summary()` — a flat summary of one run. This notebook turns on **OpenTelemetry tracing** with `StrandsTelemetry`, so you see the actual hierarchical execution: which cycle called which model invocation, which invocation triggered which tool, in order.

## The Tools

Same travel agent as demo 01 — real APIs, no injected failures.

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `search_flights(origin, destination, departure_date)` | Searches real one-way fares via the Duffel **sandbox** API | Returns up to 5 offers sorted by price |
| `get_weather(city, target_date)` | Real daily forecast via Open-Meteo (no auth) | Only covers dates within ~16 days from today |
| `book_flight(offer_id, given_name, family_name, amount, currency)` | Writes a confirmed booking to a local SQLite ledger | No paid order is ever placed |

## Doc this notebook is built from

[Strands Agents: Traces](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/) — `StrandsTelemetry`, `setup_console_exporter()`, and the documented span hierarchy:

```
Strands Agent (gen_ai.usage.total_tokens, gen_ai.user.message, gen_ai.choice, ...)
  └─ Cycle <cycle-id> (event_loop.cycle_id, gen_ai.choice.tool.result, ...)
        └─ Model invoke (gen_ai.request.model, prompt/completion, token usage)
        └─ Tool: <tool name> (gen_ai.tool.name, gen_ai.tool.call.id, tool.status)
```



In [1]:
import json
import os
from datetime import datetime, timedelta

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel  # OpenAI-compatible interface via Strands SDK
from strands.telemetry import StrandsTelemetry

import travel_tools as T

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys "
        "and add it to a .env file."
    )

# Registers a global tracer provider; the console exporter prints every span as it closes.
strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_console_exporter()

model = OpenAIModel(model_id="gpt-4o-mini")

SYSTEM_PROMPT = (
    "You are a travel assistant. Search flights, check the weather at the destination, "
    "and book the best option for the traveler without asking for confirmation. Be concise."
)

TRIP_DATE = (datetime.now() + timedelta(days=5)).strftime("%Y-%m-%d")
TRIP_PROMPT = (
    f"Book a one-way flight from JFK to MIA on {TRIP_DATE} for John Doe, "
    "and tell me if he'll need a jacket."
)

print("✅ Setup complete!")


✅ Setup complete!


## Run the agent with tracing on

`trace_attributes` tags every span this agent produces with a static session id — the documented [Custom Attribute Tracking](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/#82-custom-attribute-tracking) pattern (demo 03 builds a dynamic, per-call version with hooks).

Each span prints as JSON when it closes. Look for four span names in the output below: `invoke_agent Strands Agents` (the whole run), `execute_event_loop_cycle` (one reasoning cycle), `chat` (a model invocation), and `execute_tool <name>` (one per tool call).


In [2]:
T.init_booking_db()
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT,
              tools=[T.search_flights, T.get_weather, T.book_flight],
              trace_attributes={"session.id": "demo-02-opentelemetry-traces"})

result = agent(TRIP_PROMPT)



Tool #1: search_flights

Tool #2: get_weather
{
    "name": "chat",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0xa5437a7b5bfe466a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x37f931e3c84ff37e",
    "start_time": "2026-07-16T23:05:33.826182Z",
    "end_time": "2026-07-16T23:05:35.821094Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:33.826183+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "session.id": "demo-02-opentelemetry-traces",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.event.end_time": "2026-07-16T23:05:35.821030+00:00",
        "gen_ai.usage.prompt_tokens": 498,
        "gen_ai.usage.input_tokens": 498,
        "gen_ai.usage.completion_tokens": 71,
        "gen_ai.usage.output_tokens": 71,
        "gen_ai.usage.total_tokens": 569,
 

{
    "name": "execute_tool search_flights",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0x55abd5c282765fac",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x37f931e3c84ff37e",
    "start_time": "2026-07-16T23:05:35.822198Z",
    "end_time": "2026-07-16T23:05:36.764014Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:35.822202+00:00",
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.system": "strands-agents",
        "gen_ai.tool.name": "search_flights",
        "gen_ai.tool.call.id": "call_ivOuOfzPM6eA33i0zWZG9ox1",
        "session.id": "demo-02-opentelemetry-traces",
        "gen_ai.tool.description": "Search one-way flight offers (Duffel sandbox). Present these to the traveler to choose from.\n\nReturns:\n    A dict with `offers`: up to 5 options, each with `offer_id`, `airline`,\n    `total_amount`,

{
    "name": "execute_tool get_weather",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0xa9a4a4cfa4207b91",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x37f931e3c84ff37e",
    "start_time": "2026-07-16T23:05:35.822612Z",
    "end_time": "2026-07-16T23:05:37.319296Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:35.822614+00:00",
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.system": "strands-agents",
        "gen_ai.tool.name": "get_weather",
        "gen_ai.tool.call.id": "call_EazaaqxdfkKhNll2JmgDg464",
        "session.id": "demo-02-opentelemetry-traces",
        "gen_ai.tool.description": "Get the daily weather forecast for a city on a date (Open-Meteo), to advise on packing.\n\nReturns:\n    A dict with `city`, `temperature_max_c`, `temperature_min_c`, or an `error`.",
        "gen_ai.tool.jso

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0x37f931e3c84ff37e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf4671677adf9f090",
    "start_time": "2026-07-16T23:05:33.825994Z",
    "end_time": "2026-07-16T23:05:37.320558Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:33.825995+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "dc737ec3-7f3f-4c2d-a8ee-9ecbd6b1a574",
        "session.id": "demo-02-opentelemetry-traces",
        "gen_ai.event.end_time": "2026-07-16T23:05:37.320537+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-07-16T23:05:33.826013Z",
            "attributes": {
                "content": "[{\"text\": 


Tool #3: book_flight
{
    "name": "chat",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0xe0ecae00b4b3b75c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xa1d25b013cf207f3",
    "start_time": "2026-07-16T23:05:37.321602Z",
    "end_time": "2026-07-16T23:05:38.612120Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:37.321604+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "session.id": "demo-02-opentelemetry-traces",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.event.end_time": "2026-07-16T23:05:38.612085+00:00",
        "gen_ai.usage.prompt_tokens": 958,
        "gen_ai.usage.input_tokens": 958,
        "gen_ai.usage.completion_tokens": 50,
        "gen_ai.usage.output_tokens": 50,
        "gen_ai.usage.total_tokens": 1008,
        "gen_ai.server.ti

...

 the weather is expected to be warm, with a maximum temperature of 

31.5°C (about 88.7°F) and a minimum of 28.7°C (about 83.7°F). He won't need a jacket.

{
    "name": "chat",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0xa2395bf1e6564d07",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xa817f051fe7d512b",
    "start_time": "2026-07-16T23:05:38.617770Z",
    "end_time": "2026-07-16T23:05:40.347708Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:38.617771+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "session.id": "demo-02-opentelemetry-traces",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.event.end_time": "2026-07-16T23:05:40.347664+00:00",
        "gen_ai.usage.prompt_tokens": 1057,
        "gen_ai.usage.input_tokens": 1057,
        "gen_ai.usage.completion_tokens": 94,
        "gen_ai.usage.output_tokens": 94,
        "gen_ai.usage.total_tokens": 1151,
        "gen_ai.server.time_to_first_token": 

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0xa817f051fe7d512b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf4671677adf9f090",
    "start_time": "2026-07-16T23:05:38.617414Z",
    "end_time": "2026-07-16T23:05:40.349405Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:38.617417+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "d22bcbff-80e7-4685-aa8a-17bcc58c3b64",
        "session.id": "demo-02-opentelemetry-traces",
        "event_loop.parent_cycle_id": "6c516f9f-d277-4360-bbc9-45b0a3aad7c7",
        "gen_ai.event.end_time": "2026-07-16T23:05:40.349371+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-07-16T23:05:38.

{
    "name": "invoke_agent Strands Agents",
    "context": {
        "trace_id": "0x0be24668ae0502080fbeb5fd51331855",
        "span_id": "0xf4671677adf9f090",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-16T23:05:33.825783Z",
    "end_time": "2026-07-16T23:05:40.350467Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-16T23:05:33.825792+00:00",
        "gen_ai.operation.name": "invoke_agent",
        "gen_ai.system": "strands-agents",
        "gen_ai.agent.name": "Strands Agents",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.agent.tools": "[\"search_flights\", \"get_weather\", \"book_flight\"]",
        "session.id": "demo-02-opentelemetry-traces",
        "system_prompt": "You are a travel assistant. Search flights, check the weather at the destination, and book the best option for the traveler without asking for confirmation.

## Walk the span tree

The console exporter above printed every span. This cell confirms, from the same run, which span names actually appeared — the four levels the [Trace Structure](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/#4-trace-structure) doc describes.


In [3]:
summary = result.metrics.get_summary()
print(f"Cycles: {summary['total_cycles']}")
print(f"Tools called: {list(summary['tool_usage'].keys())}")
print()
print("Final response:")
print(result.message["content"][0]["text"])


Cycles: 3
Tools called: ['search_flights', 'get_weather', 'book_flight']

Final response:
John Doe's flight from JFK to MIA on July 21, 2026, has been booked with Iberia for $89.09. His booking reference is BK-5VMVEZ.

In Miami, the weather is expected to be warm, with a maximum temperature of 31.5°C (about 88.7°F) and a minimum of 28.7°C (about 83.7°F). He won't need a jacket.


## Ground truth


In [4]:
print(json.dumps(T.query_booked_offers(), indent=2))


[
  {
    "booking_reference": "BK-5VMVEZ",
    "offer_id": "off_0000B8PNugJzTZ165VMvEz",
    "passenger": "John Doe",
    "amount": "89.09",
    "currency": "USD"
  }
]


## Key takeaways

- **Demo 01's metrics** are a flat summary: totals and per-tool aggregates for the whole run.
- **Traces** are the actual timeline: an `invoke_agent` span contains `execute_event_loop_cycle` spans, each containing one `chat` (model) span and zero or more `execute_tool` spans — exactly the hierarchy the docs describe, not a custom shape.
- `trace_attributes` tags EVERY span in this run with the same static metadata (here, a session id). Demo 03 shows a dynamic version: tagging only the spans where a business rule fires.

## References

- [Strands Agents: Traces](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/)
- Previous: [01 - Agent Metrics](../01-agent-metrics/) · Next: [03 - Custom Trace Attributes](../03-custom-trace-attributes/)
